# 01 Proposed Method: Backbone Comparison Experiments
**Paper**: XDenseQNet: Hybrid Quantum Neural Network for Skin Lesion Classification
**Authors**: Shehroz Tariq, Umair Inayat, Fatima Zulfiqar, Rehan Raza*, Muhammad Arif
**Description**: Trains 8 CNN backbones with CBAM attention + parameterized quantum circuit (4Q-2L and 6Q-3L configurations) on MSLD v2.0. Produces comparison tables across backbone and quantum settings.
**Runtime**: ~4-6 hours on a single GPU (NVIDIA T4 or better)


In [ ]:
# !pip install -r ../requirements.txt


In [ ]:
import random, numpy as np, torch
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


In [ ]:
# -*- coding: utf-8 -*-
"""
Transfer Learning + Hybrid QNN
11 Backbones × 2 Quantum Configurations

"""

import pennylane as qml
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as transforms
import torchvision.models as models
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.preprocessing import label_binarize
import os
import random
from tqdm import tqdm
import json
import warnings
import shutil
import albumentations as A
from albumentations.pytorch import ToTensorV2
import seaborn as sns
import pandas as pd
import copy
import time
from collections import defaultdict
import timm  # For ConvNeXt, EfficientNetV2, Swin Transformer

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8')

print("All dependencies imported successfully!")
print(f"PyTorch version: {torch.__version__}")
print(f"Timm version: {timm.__version__}")
print(f"Device: {'CUDA' if torch.cuda.is_available() else 'CPU'}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
class QuantumConfig:
    """Configuration for quantum parameters"""
    def __init__(self, n_qubits, n_layers, name):
        self.n_qubits = n_qubits
        self.n_layers = n_layers
        self.name = name
        self.feature_dim = n_qubits

class Config:
    """Complete configuration for all 11 backbones × 2 quantum configs"""

    def __init__(self, quantum_config):
        # Paths
        self.original_dataset_path = "Monkeypox Skin Image Dataset"
        self.processed_dataset_path = "processed_balanced_dataset"
        self.results_dir = f"all_11_backbones_{quantum_config.name}_results"
        os.makedirs(self.results_dir, exist_ok=True)

        # Image settings
        self.image_size = (224, 224)
        self.batch_size = 16

        # Classes
        self.num_classes = 4
        self.class_names = ['Normal', 'Monkeypox', 'Chickenpox', 'Measles']

        # BALANCED AUGMENTATION - All classes to ~550
        self.target_samples_per_class = 550

        # Data splits
        self.train_ratio = 0.7
        self.val_ratio = 0.15
        self.test_ratio = 0.15
        self.random_state = 42

        # Transfer learning
        self.pretrain_epochs = 30
        self.pretrain_lr = 0.001
        self.freeze_last_layer = True

        # Main training
        self.num_epochs = 80
        self.patience = 20
        self.learning_rate = 0.0008
        self.weight_decay = 1e-3

        # Quantum configuration
        self.n_qubits = quantum_config.n_qubits
        self.n_layers = quantum_config.n_layers
        self.feature_dim = quantum_config.feature_dim
        self.quantum_name = quantum_config.name

        # Model
        self.dropout_rate = 0.5
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

        # Training features
        self.use_label_smoothing = True
        self.label_smoothing = 0.1
        self.use_mixup = True
        self.mixup_alpha = 0.2
        self.use_attention = True
        self.use_cosine_annealing = True
        self.cosine_T_0 = 10
        self.cosine_T_mult = 2
        self.focal_gamma = 2.0
        self.unfreeze_epoch = 15
        self.unfreeze_lr_multiplier = 0.01

        # 11 BACKBONES
        self.backbones = {
            # Original 6
            'resnet18': {
                'name': 'ResNet18',
                'features': 512,
                'type': 'torchvision'
            },
            'resnet50': {
                'name': 'ResNet50',
                'features': 2048,
                'type': 'torchvision'
            },
            'densenet121': {
                'name': 'DenseNet121',
                'features': 1024,
                'type': 'torchvision'
            },
            'efficientnet_b0': {
                'name': 'EfficientNet-B0',
                'features': 1280,
                'type': 'torchvision'
            },
            'mobilenet_v2': {
                'name': 'MobileNet-V2',
                'features': 1280,
                'type': 'torchvision'
            },
            'vgg16': {
                'name': 'VGG16',
                'features': 4096,
                'type': 'torchvision'
            },
            # New 5
            'convnext_tiny': {
                'name': 'ConvNeXt-Tiny',
                'features': 768,
                'type': 'timm'
            },

            'swin_tiny': {
                'name': 'Swin-Tiny',
                'features': 768,
                'type': 'timm'
            }
        }

        self._display_info()

    def _display_info(self):
        print("\n" + "="*80)
        print(f"TRANSFER LEARNING + HYBRID QNN - {self.quantum_name}")
        print("="*80)
        print(f"Classes: {', '.join(self.class_names)}")
        print(f"\nBalanced Augmentation Strategy (Training only):")
        print(f"  • ALL classes → {self.target_samples_per_class} images")
        print(f"  • Validation/Test: NO augmentation (original)")
        print(f"\nQuantum Configuration:")
        print(f"  • Qubits: {self.n_qubits}")
        print(f"  • Layers: {self.n_layers}")
        print(f"\nBackbones ({len(self.backbones)}):")
        for key, info in self.backbones.items():
            print(f"  • {info['name']} ({info['features']} features) [{info['type']}]")
        print(f"\nTransfer Learning:")
        print(f"  • Fine-tuning epochs: {self.pretrain_epochs}")
        print(f"  • Main training epochs: {self.num_epochs}")
        print(f"  • Freeze last layer: {self.freeze_last_layer}")
        print(f"\nFeatures:")
        print(f"  • Attention: {'ON' if self.use_attention else 'OFF'}")
        print(f"  • Label Smoothing: {self.label_smoothing}")
        print(f"  • Mixup: alpha={self.mixup_alpha}")
        print(f"  • Inference Time Measurement: ENABLED")
        print("="*80)

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

# Create 2 quantum configurations
quantum_configs = {
    'Q4L2': QuantumConfig(n_qubits=4, n_layers=2, name="4Q_2L"),
    'Q6L3': QuantumConfig(n_qubits=6, n_layers=3, name="6Q_3L")
}

print("\n" + "="*80)
print("QUANTUM CONFIGURATIONS")
print("="*80)
print("Experiment 1: 4 Qubits, 2 Layers (Q4L2)")
print("Experiment 2: 6 Qubits, 3 Layers (Q6L3)")
print("="*80)

In [ ]:
class FocalLoss(nn.Module):
    """Focal Loss for class imbalance"""
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = nn.functional.cross_entropy(
            inputs, targets, reduction='none', weight=self.alpha
        )
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        if self.reduction == 'mean':
            return focal_loss.mean()
        return focal_loss

class LabelSmoothingFocalLoss(nn.Module):
    """Focal Loss + Label Smoothing"""
    def __init__(self, num_classes, alpha=None, gamma=2.0, smoothing=0.1, reduction='mean'):
        super(LabelSmoothingFocalLoss, self).__init__()
        self.num_classes = num_classes
        self.alpha = alpha
        self.gamma = gamma
        self.smoothing = smoothing
        self.reduction = reduction

    def forward(self, inputs, targets):
        with torch.no_grad():
            smooth_labels = torch.zeros_like(inputs)
            smooth_labels.fill_(self.smoothing / (self.num_classes - 1))
            smooth_labels.scatter_(1, targets.unsqueeze(1), 1.0 - self.smoothing)

        log_probs = torch.log_softmax(inputs, dim=1)
        loss = (-smooth_labels * log_probs).sum(dim=1)
        pt = torch.exp(-loss)
        focal_loss = ((1 - pt) ** self.gamma) * loss

        if self.alpha is not None:
            class_weights = self.alpha[targets]
            focal_loss = focal_loss * class_weights

        if self.reduction == 'mean':
            return focal_loss.mean()
        return focal_loss

print("✓ Loss functions loaded!")

In [ ]:
class BalancedDataBalancer:
    """Balanced augmentation - all classes to target"""

    def __init__(self, config):
        self.config = config
        self.augmentation_transform = self._create_augmentation_transform()

    def _create_augmentation_transform(self):
        return A.Compose([
            A.Resize(self.config.image_size[0], self.config.image_size[1]),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.2),
            A.Rotate(limit=20, p=0.5),
            A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.15, rotate_limit=20, p=0.5),
            A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
            A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=20, p=0.3),
            A.OneOf([
                A.GaussNoise(var_limit=(10.0, 50.0), p=0.5),
                A.GaussianBlur(blur_limit=(3, 5), p=0.5),
            ], p=0.3),
            A.CoarseDropout(max_holes=8, max_height=16, max_width=16,
                          min_holes=1, min_height=8, min_width=8, fill_value=0, p=0.2),
            A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ToTensorV2()
        ])

    def generate_augmented_images(self, image_path, num_augmentations):
        try:
            image = Image.open(image_path).convert('RGB')
            image = image.resize(self.config.image_size, Image.LANCZOS)
            image_np = np.array(image)

            augmented_images = []
            for i in range(num_augmentations):
                augmented = self.augmentation_transform(image=image_np)
                img_tensor = augmented['image']

                mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
                std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
                img_denorm = img_tensor * std + mean
                img_denorm = torch.clamp(img_denorm, 0, 1)
                img_np = img_denorm.permute(1, 2, 0).numpy()
                img_pil = Image.fromarray((img_np * 255).astype(np.uint8))
                augmented_images.append(img_pil)

            return augmented_images
        except:
            return []

def get_transforms(config):
    """Training: Strong augmentation, Val/Test: None"""
    train_transform = A.Compose([
        A.Resize(config.image_size[0], config.image_size[1]),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.2),
        A.Rotate(limit=15, p=0.5),
        A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.15, rotate_limit=15, p=0.5),
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
        A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=20, p=0.3),
        A.OneOf([
            A.GaussNoise(var_limit=(10.0, 50.0), p=0.5),
            A.GaussianBlur(blur_limit=(3, 5), p=0.5),
        ], p=0.3),
        A.CoarseDropout(max_holes=8, max_height=16, max_width=16,
                      min_holes=1, min_height=8, min_width=8, fill_value=0, p=0.3),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2()
    ])

    val_test_transform = A.Compose([
        A.Resize(config.image_size[0], config.image_size[1]),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2()
    ])

    return train_transform, val_test_transform, val_test_transform

print("✓ Balanced data augmentation loaded!")

In [ ]:
def organize_balanced_dataset(source_path, target_path, config):
    """Organize with BALANCED augmentation - all classes to ~550"""

    print(f"\nSETTING UP DATASET")
    print("="*70)
    print("ORGANIZING DATASET WITH BALANCED AUGMENTATION")
    print("="*70)

    if os.path.exists(target_path):
        shutil.rmtree(target_path)
    os.makedirs(target_path, exist_ok=True)

    # Check original structure
    print("CHECKING DATASET STRUCTURE")
    print("="*60)

    class_counts = {}
    total_images = 0

    for item in os.listdir(source_path):
        item_path = os.path.join(source_path, item)
        if os.path.isdir(item_path):
            images = [f for f in os.listdir(item_path)
                     if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff'))]
            count = len(images)
            class_counts[item] = count
            total_images += count
            print(f"  {item}: {count} images")

    print(f"Total: {total_images} images")

    # Create splits
    for split in ['train', 'val', 'test']:
        for class_name in config.class_names:
            os.makedirs(os.path.join(target_path, split, class_name.lower()), exist_ok=True)

    # Collect images
    all_class_images = {}
    for class_name in config.class_names:
        found_folder = None
        for folder in os.listdir(source_path):
            folder_path = os.path.join(source_path, folder)
            if os.path.isdir(folder_path):
                if class_name.lower() in folder.lower():
                    found_folder = folder
                    break

        if found_folder:
            folder_path = os.path.join(source_path, found_folder)
            images = [os.path.join(folder_path, f) for f in os.listdir(folder_path)
                     if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff'))]
            all_class_images[class_name] = images

    # Split and copy
    split_counts = {'train': {}, 'val': {}, 'test': {}}

    for class_name, images in all_class_images.items():
        random.shuffle(images)
        total = len(images)
        train_size = int(total * config.train_ratio)
        val_size = int(total * config.val_ratio)

        splits_data = {
            'train': images[:train_size],
            'val': images[train_size:train_size + val_size],
            'test': images[train_size + val_size:]
        }

        for split, split_images in splits_data.items():
            split_class_dir = os.path.join(target_path, split, class_name.lower())
            for i, img_path in enumerate(split_images):
                img_name = f"{class_name.lower()}_{split}_{i:04d}.jpg"
                shutil.copy2(img_path, os.path.join(split_class_dir, img_name))
            split_counts[split][class_name] = len(split_images)

    # Display original split counts
    print(f"Original split counts:")
    for split in ['train', 'val', 'test']:
        print(f"  {split.upper()}:")
        for class_name in config.class_names:
            print(f"    {class_name}: {split_counts[split][class_name]}")

    # BALANCED AUGMENTATION for training
    print(f"\nAugmenting TRAINING data:")
    data_balancer = BalancedDataBalancer(config)

    for class_name in config.class_names:
        current = split_counts['train'][class_name]
        target = config.target_samples_per_class
        target_dir = os.path.join(target_path, 'train', class_name.lower())

        needed = target - current

        if needed > 0:
            original_files = [f for f in os.listdir(target_dir)
                            if f.endswith(('.jpg', '.jpeg', '.png'))][:15]

            total_generated = 0
            per_image = max(1, needed // len(original_files))

            for idx, img_file in enumerate(original_files):
                if total_generated >= needed:
                    break
                img_path = os.path.join(target_dir, img_file)
                aug_images = data_balancer.generate_augmented_images(img_path, per_image)

                for j, aug_img in enumerate(aug_images):
                    if total_generated >= needed:
                        break
                    aug_filename = f"{class_name.lower()}_train_aug_{idx:04d}_{j:02d}.jpg"
                    aug_img.save(os.path.join(target_dir, aug_filename), 'JPEG', quality=95)
                    total_generated += 1

            final_count = current + total_generated
            print(f"    Augmenting {class_name}: {current} -> {final_count} (+{total_generated})")
            split_counts['train'][class_name] = final_count
        else:
            print(f"    {class_name}: {current} (already at target)")

    # Display final counts
    print(f"\nFinal split counts:")
    for split in ['train', 'val', 'test']:
        total = sum(split_counts[split].values())
        print(f"  {split.upper()}:")
        for class_name in config.class_names:
            print(f"    {class_name}: {split_counts[split][class_name]}")
        print(f"    Total: {total}")

    print("Dataset organization completed!")
    print("\nDataset ready!")
    return True

# This will be run once, shared across both quantum configs
print("Dataset organization function loaded!")
print("Will organize dataset before training...")

In [ ]:
class MultiClassDataset(Dataset):
    def __init__(self, data_dir, transform=None, config=None, split_name="train"):
        self.data_dir = data_dir
        self.transform = transform
        self.config = config
        self.split_name = split_name

        self.images = []
        self.labels = []
        self.class_counts = {}

        class_dirs = ['normal', 'monkeypox', 'chickenpox', 'measles']

        for class_idx, class_name in enumerate(class_dirs):
            class_path = os.path.join(data_dir, class_name)
            if os.path.exists(class_path):
                class_images = [os.path.join(class_path, f) for f in os.listdir(class_path)
                              if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff'))]
                self.images.extend(class_images)
                self.labels.extend([class_idx] * len(class_images))
                self.class_counts[class_idx] = len(class_images)

        print(f"  {split_name.upper()}: {len(self.images)} images")
        for idx, name in enumerate(['Normal', 'Monkeypox', 'Chickenpox', 'Measles']):
            print(f"    {name}: {self.class_counts.get(idx, 0)}")

    def get_class_weights(self):
        if not self.class_counts:
            return None
        total = sum(self.class_counts.values())
        num_classes = len(self.class_counts)
        weights = [total / (num_classes * self.class_counts.get(i, 1)) for i in range(num_classes)]
        return torch.FloatTensor(weights)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = self.images[idx]
        label = self.labels[idx]

        try:
            image = Image.open(img_path).convert('RGB')
            image = image.resize(self.config.image_size, Image.LANCZOS)
            image = np.array(image)

            if self.transform:
                transformed = self.transform(image=image)
                image = transformed['image']

            return image, label
        except:
            image = np.zeros((self.config.image_size[0], self.config.image_size[1], 3), dtype=np.uint8)
            if self.transform:
                transformed = self.transform(image=image)
                image = transformed['image']
            return image, label

def create_dataloaders(config):
    print(f"\nCREATING DATA LOADERS")
    print("="*60)

    train_transform, val_transform, test_transform = get_transforms(config)

    train_dataset = MultiClassDataset(
        os.path.join(config.processed_dataset_path, 'train'),
        transform=train_transform, config=config, split_name='train'
    )

    val_dataset = MultiClassDataset(
        os.path.join(config.processed_dataset_path, 'val'),
        transform=val_transform, config=config, split_name='val'
    )

    test_dataset = MultiClassDataset(
        os.path.join(config.processed_dataset_path, 'test'),
        transform=test_transform, config=config, split_name='test'
    )

    train_loader = DataLoader(train_dataset, batch_size=config.batch_size,
                             shuffle=True, num_workers=0, pin_memory=True, drop_last=True)
    val_loader = DataLoader(val_dataset, batch_size=config.batch_size,
                           shuffle=False, num_workers=0, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=config.batch_size,
                            shuffle=False, num_workers=0, pin_memory=True)

    return {
        'train_loader': train_loader, 'val_loader': val_loader, 'test_loader': test_loader,
        'train_dataset': train_dataset, 'val_dataset': val_dataset, 'test_dataset': test_dataset
    }

print("✓ Dataset class loaded!")

In [ ]:
class ChannelAttention(nn.Module):
    def __init__(self, in_channels, reduction=16):
        super(ChannelAttention, self).__init__()
        reduced = max(in_channels // reduction, 8)
        self.fc = nn.Sequential(
            nn.Linear(in_channels, reduced, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(reduced, in_channels, bias=False)
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        out = self.fc(x)
        out = self.sigmoid(out)
        return x * out

class SpatialAttention(nn.Module):
    def __init__(self, in_channels):
        super(SpatialAttention, self).__init__()
        self.fc = nn.Sequential(
            nn.Linear(in_channels, in_channels // 4),
            nn.ReLU(inplace=True),
            nn.Linear(in_channels // 4, in_channels),
            nn.Sigmoid()
        )

    def forward(self, x):
        attention = self.fc(x)
        return x * attention

class CBAM(nn.Module):
    def __init__(self, in_channels, reduction=16):
        super(CBAM, self).__init__()
        self.channel_attention = ChannelAttention(in_channels, reduction)
        self.spatial_attention = SpatialAttention(in_channels)

    def forward(self, x):
        x = self.channel_attention(x)
        x = self.spatial_attention(x)
        return x

print("✓ Attention module (CBAM) loaded!")

In [ ]:
class UniversalCNNExtractor(nn.Module):
    """Universal feature extractor for all 11 backbones"""

    def __init__(self, backbone_key, config, num_classes=4):
        super(UniversalCNNExtractor, self).__init__()

        self.backbone_key = backbone_key
        self.config = config
        self.num_classes = num_classes

        backbone_info = config.backbones[backbone_key]
        self.backbone_name = backbone_info['name']
        self.backbone_features = backbone_info['features']
        self.backbone_type = backbone_info['type']

        # Load backbone
        if self.backbone_type == 'torchvision':
            self.backbone = self._load_torchvision_backbone()
        else:  # timm
            self.backbone = self._load_timm_backbone()

        # Freeze and configure
        self._configure_backbone()

        self.adaptive_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.feature_normalizer = nn.Sequential(
            nn.BatchNorm1d(self.backbone_features),
            nn.ReLU()
        )
        self.temp_classifier = nn.Linear(self.backbone_features, num_classes)

        trainable = sum(p.numel() for p in self.backbone.parameters() if p.requires_grad)
        print(f"  {self.backbone_name}: {self.backbone_features} features, {trainable:,} trainable params")

    def _load_torchvision_backbone(self):
        if self.backbone_key == 'resnet18':
            model = models.resnet18(pretrained=True)
            return nn.Sequential(*list(model.children())[:-1])
        elif self.backbone_key == 'resnet50':
            model = models.resnet50(pretrained=True)
            return nn.Sequential(*list(model.children())[:-1])
        elif self.backbone_key == 'densenet121':
            model = models.densenet121(pretrained=True)
            model.classifier = nn.Identity()
            return model
        elif self.backbone_key == 'efficientnet_b0':
            model = models.efficientnet_b0(pretrained=True)
            model.classifier = nn.Identity()
            return model
        elif self.backbone_key == 'mobilenet_v2':
            model = models.mobilenet_v2(pretrained=True)
            model.classifier = nn.Identity()
            return model
        elif self.backbone_key == 'vgg16':
            model = models.vgg16(pretrained=True)
            model.classifier = nn.Sequential(*list(model.classifier.children())[:-1])
            return model

    def _load_timm_backbone(self):
        if self.backbone_key == 'convnext_tiny':
            model = timm.create_model('convnext_tiny', pretrained=True, num_classes=0)
        elif self.backbone_key == 'inception_resnet_v2':
            model = timm.create_model('inception_resnet_v2', pretrained=True, num_classes=0)
        elif self.backbone_key == 'efficientnetv2_s':
            model = timm.create_model('tf_efficientnetv2_s', pretrained=True, num_classes=0)
        elif self.backbone_key == 'xception':
            model = timm.create_model('xception', pretrained=True, num_classes=0)
        elif self.backbone_key == 'swin_tiny':
            model = timm.create_model('swin_tiny_patch4_window7_224', pretrained=True, num_classes=0)
        return model

    def _configure_backbone(self):
        # Freeze all initially
        for param in self.backbone.parameters():
            param.requires_grad = False

        # Unfreeze specific layers
        if self.backbone_type == 'torchvision':
            if 'resnet' in self.backbone_key:
                for param in self.backbone[-1].parameters():
                    param.requires_grad = True
            elif 'densenet' in self.backbone_key:
                for param in self.backbone.features.denseblock4.parameters():
                    param.requires_grad = True
            elif 'efficientnet' in self.backbone_key or 'mobilenet' in self.backbone_key:
                total_blocks = len(self.backbone.features)
                for i in range(total_blocks - 5, total_blocks - 1):
                    for param in self.backbone.features[i].parameters():
                        param.requires_grad = True
            elif 'vgg' in self.backbone_key:
                for param in list(self.backbone.features.parameters())[-10:]:
                    param.requires_grad = True
        else:  # timm
            # Unfreeze last stages
            if hasattr(self.backbone, 'stages'):
                for param in self.backbone.stages[-1].parameters():
                    param.requires_grad = True
            elif hasattr(self.backbone, 'blocks'):
                for param in self.backbone.blocks[-2:].parameters():
                    param.requires_grad = True
            elif hasattr(self.backbone, 'layers'):
                for param in self.backbone.layers[-1].parameters():
                    param.requires_grad = True

        # Freeze last layer if required
        if self.config.freeze_last_layer:
            self._freeze_last_layer()

    def _freeze_last_layer(self):
        if self.backbone_type == 'torchvision':
            if 'resnet' in self.backbone_key:
                for param in list(self.backbone[-1].parameters())[-2:]:
                    param.requires_grad = False
            elif 'densenet' in self.backbone_key:
                for param in list(self.backbone.features.parameters())[-2:]:
                    param.requires_grad = False
            elif 'efficientnet' in self.backbone_key or 'mobilenet' in self.backbone_key:
                for param in self.backbone.features[-1].parameters():
                    param.requires_grad = False
        else:
            # Freeze last block in timm models
            if hasattr(self.backbone, 'stages'):
                for param in list(self.backbone.stages[-1].parameters())[-2:]:
                    param.requires_grad = False

    def forward(self, x, return_features_only=False):
        features = self.backbone(x)
        if len(features.shape) == 4:
            features = self.adaptive_pool(features)
        features = features.view(features.size(0), -1)
        features = self.feature_normalizer(features)

        if return_features_only:
            return features
        else:
            return self.temp_classifier(features)

    def get_output_dim(self):
        return self.backbone_features

    def unfreeze_for_full_training(self):
        for param in self.backbone.parameters():
            param.requires_grad = True
        if self.config.freeze_last_layer:
            self._freeze_last_layer()

print("✓ Universal CNN Extractor loaded for all 11 backbones!")

In [ ]:
def create_quantum_circuit(config):
    """Create quantum circuit based on config"""
    n_qubits = config.n_qubits
    n_layers = config.n_layers

    dev = qml.device("default.qubit", wires=n_qubits)

    def quantum_circuit(inputs, weights):
        # Encoding
        for i in range(min(len(inputs), n_qubits)):
            qml.RY(inputs[i] * np.pi, wires=i)

        # Quantum layers
        for layer in range(n_layers):
            for i in range(n_qubits):
                qml.RX(weights[layer, i, 0], wires=i)
                qml.RY(weights[layer, i, 1], wires=i)
                qml.RZ(weights[layer, i, 2], wires=i)

            for i in range(n_qubits):
                qml.CNOT(wires=[i, (i + 1) % n_qubits])

            if layer % 2 == 1 and n_qubits > 1:
                for i in range(0, n_qubits - 1, 2):
                    qml.CNOT(wires=[i, i + 1])

        return [qml.expval(qml.PauliZ(wires=i)) for i in range(n_qubits)]

    qnode = qml.QNode(quantum_circuit, dev, interface="torch")

    print(f"✓ Quantum circuit: {n_qubits} qubits, {n_layers} layers")
    return qnode

print("✓ Quantum circuit factory loaded!")

In [ ]:
class HybridQNN(nn.Module):
    """Hybrid Quantum-Classical Neural Network"""

    def __init__(self, backbone_key, config, qnode):
        super(HybridQNN, self).__init__()

        self.config = config
        self.backbone_key = backbone_key
        self.num_classes = config.num_classes
        self.qnode = qnode

        # Feature extractor
        self.feature_extractor = UniversalCNNExtractor(backbone_key, config, config.num_classes)
        self.backbone_output_dim = self.feature_extractor.get_output_dim()

        # Attention
        if config.use_attention:
            self.attention = CBAM(self.backbone_output_dim, reduction=16)
        else:
            self.attention = None

        # Quantum weights
        self.quantum_weights = nn.Parameter(
            torch.randn(config.n_layers, config.n_qubits, 3) * 0.1
        )

        # Quantum preprocessing
        self.quantum_preprocessing = nn.Sequential(
            nn.Linear(self.backbone_output_dim, config.feature_dim),
            nn.Tanh()
        )

        # Classifier
        self.classifier = nn.Sequential(
            nn.Linear(self.backbone_output_dim + config.n_qubits, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(config.dropout_rate),

            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(config.dropout_rate),

            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(config.dropout_rate),

            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(config.dropout_rate * 0.5),

            nn.Linear(64, self.num_classes)
        )

        self.class_weights = None
        self._initialize_weights()

    def _initialize_weights(self):
        for m in self.classifier.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.constant_(m.bias, 0)
        for m in self.quantum_preprocessing.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.constant_(m.bias, 0)

    def set_class_weights(self, class_weights):
        if class_weights is not None:
            self.class_weights = class_weights.to(self.config.device)

    def forward(self, x, pretrain_mode=False):
        if pretrain_mode:
            return self.feature_extractor(x, return_features_only=False)

        # Full hybrid pipeline
        classical_features = self.feature_extractor(x, return_features_only=True)

        if self.attention is not None:
            classical_features = self.attention(classical_features)

        # Quantum processing
        quantum_input = self.quantum_preprocessing(classical_features)

        quantum_outputs = []
        for i in range(quantum_input.size(0)):
            try:
                q_input_cpu = quantum_input[i].detach().cpu()
                q_weights_cpu = self.quantum_weights.detach().cpu()
                quantum_out = self.qnode(q_input_cpu, q_weights_cpu)
                quantum_tensor = torch.stack([q.clone().detach() for q in quantum_out]).float()
                quantum_outputs.append(quantum_tensor)
            except:
                quantum_outputs.append(torch.zeros(self.config.n_qubits, dtype=torch.float32))

        quantum_features = torch.stack(quantum_outputs).to(x.device)

        # Combine and classify
        combined = torch.cat([classical_features, quantum_features], dim=1)
        output = self.classifier(combined)
        return output

    def get_loss_function(self):
        if self.config.use_label_smoothing:
            return LabelSmoothingFocalLoss(
                num_classes=self.num_classes,
                alpha=self.class_weights,
                gamma=self.config.focal_gamma,
                smoothing=self.config.label_smoothing
            )
        else:
            return FocalLoss(alpha=self.class_weights, gamma=self.config.focal_gamma)

print("✓ Hybrid QNN model loaded!")

In [ ]:
def mixup_data(x, y, alpha=0.2):
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1
    batch_size = x.size(0)
    index = torch.randperm(batch_size).to(x.device)
    mixed_x = lam * x + (1 - lam) * x[index, :]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

class EarlyStopping:
    def __init__(self, patience=15, min_delta=0.001):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = float('inf')
        self.best_weights = None

    def __call__(self, val_loss, model):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
            self.best_weights = copy.deepcopy(model.state_dict())
        else:
            self.counter += 1

        if self.counter >= self.patience:
            if self.best_weights:
                model.load_state_dict(self.best_weights)
            return True
        return False

def fine_tune_backbone(model, train_loader, val_loader, config, backbone_name):
    """Phase 1: Fine-tune - 30 epochs"""
    print(f"\nPHASE 1: FINE-TUNING {backbone_name}")
    print("="*80)

    model.to(config.device)
    criterion = nn.CrossEntropyLoss()

    optimizer = optim.Adam([
        {'params': model.feature_extractor.backbone.parameters(), 'lr': config.pretrain_lr * 0.1},
        {'params': model.feature_extractor.feature_normalizer.parameters(), 'lr': config.pretrain_lr},
        {'params': model.feature_extractor.temp_classifier.parameters(), 'lr': config.pretrain_lr}
    ], weight_decay=config.weight_decay)

    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)
    best_val_acc = 0.0

    for epoch in range(config.pretrain_epochs):
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0

        pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{config.pretrain_epochs}', leave=False)
        for data, target in pbar:
            data, target = data.to(config.device), target.to(config.device).long()
            optimizer.zero_grad()
            output = model(data, pretrain_mode=True)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            _, predicted = torch.max(output.data, 1)
            train_correct += (predicted == target).sum().item()
            train_total += target.size(0)
            pbar.set_postfix({'Loss': f'{loss.item():.4f}', 'Acc': f'{100.*train_correct/train_total:.1f}%'})

        train_loss /= len(train_loader)
        train_acc = 100. * train_correct / train_total

        # Validation
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for data, target in val_loader:
                data, target = data.to(config.device), target.to(config.device).long()
                output = model(data, pretrain_mode=True)
                loss = criterion(output, target)
                val_loss += loss.item()
                _, predicted = torch.max(output.data, 1)
                val_correct += (predicted == target).sum().item()
                val_total += target.size(0)

        val_loss /= len(val_loader)
        val_acc = 100. * val_correct / val_total
        scheduler.step(val_loss)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            print(f"Epoch {epoch+1:2d} | Train: {train_loss:.4f} {train_acc:.1f}% | Val: {val_loss:.4f} {val_acc:.1f}% ★")
        else:
            print(f"Epoch {epoch+1:2d} | Train: {train_loss:.4f} {train_acc:.1f}% | Val: {val_loss:.4f} {val_acc:.1f}%")

    print(f"Fine-tuning complete! Best Val Acc: {best_val_acc:.2f}%")
    return model

def train_hybrid_qnn(model, train_loader, val_loader, config, backbone_name):
    """Phase 2: Train full hybrid QNN"""
    print(f"\nPHASE 2: TRAINING HYBRID QNN - {backbone_name}")
    print("="*80)

    model.to(config.device)

    train_dataset = train_loader.dataset
    class_weights = train_dataset.get_class_weights()
    if class_weights is not None:
        model.set_class_weights(class_weights)

    criterion = model.get_loss_function()

    param_groups = [
        {'params': model.feature_extractor.backbone.parameters(),
         'lr': config.learning_rate * 0.01, 'name': 'backbone'},
        {'params': [model.quantum_weights],
         'lr': config.learning_rate, 'name': 'quantum'},
        {'params': list(model.feature_extractor.feature_normalizer.parameters()) +
                  list(model.quantum_preprocessing.parameters()) +
                  list(model.classifier.parameters()) +
                  (list(model.attention.parameters()) if model.attention else []),
         'lr': config.learning_rate, 'name': 'other'}
    ]

    optimizer = optim.AdamW(param_groups, weight_decay=config.weight_decay)

    if config.use_cosine_annealing:
        scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
            optimizer, T_0=config.cosine_T_0, T_mult=config.cosine_T_mult, eta_min=1e-7
        )
    else:
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

    early_stopping = EarlyStopping(patience=config.patience)
    best_val_acc = 0.0
    backbone_unfrozen = False

    for epoch in range(config.num_epochs):
        if epoch == config.unfreeze_epoch and not backbone_unfrozen:
            print(f"*** UNFREEZING BACKBONE ***")
            model.feature_extractor.unfreeze_for_full_training()
            backbone_unfrozen = True
            for pg in optimizer.param_groups:
                if pg.get('name') == 'backbone':
                    pg['lr'] = config.learning_rate * config.unfreeze_lr_multiplier

        # Training
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0

        pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}', leave=False)
        for data, target in pbar:
            data, target = data.to(config.device), target.to(config.device).long()
            optimizer.zero_grad()

            if config.use_mixup and np.random.random() > 0.5:
                data, targets_a, targets_b, lam = mixup_data(data, target, config.mixup_alpha)
                output = model(data, pretrain_mode=False)
                loss = mixup_criterion(criterion, output, targets_a, targets_b, lam)
            else:
                output = model(data, pretrain_mode=False)
                loss = criterion(output, target)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            train_loss += loss.item()
            _, predicted = torch.max(output.data, 1)
            train_correct += (predicted == target).sum().item()
            train_total += target.size(0)
            pbar.set_postfix({'Loss': f'{loss.item():.4f}', 'Acc': f'{100.*train_correct/train_total:.1f}%'})

        train_loss /= len(train_loader)
        train_acc = 100. * train_correct / train_total

        # Validation
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for data, target in val_loader:
                data, target = data.to(config.device), target.to(config.device).long()
                output = model(data, pretrain_mode=False)
                loss = criterion(output, target)
                val_loss += loss.item()
                _, predicted = torch.max(output.data, 1)
                val_correct += (predicted == target).sum().item()
                val_total += target.size(0)

        val_loss /= len(val_loader)
        val_acc = 100. * val_correct / val_total

        if config.use_cosine_annealing:
            scheduler.step()
        else:
            scheduler.step(val_loss)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            status = "UNFROZEN" if backbone_unfrozen else "FROZEN"
            print(f"Epoch {epoch+1:3d} | Train: {train_loss:.4f} {train_acc:.1f}% | Val: {val_loss:.4f} {val_acc:.1f}% [{status}] ★")

        if early_stopping(val_loss, model):
            print(f"Early stopping at epoch {epoch+1}")
            break

    print(f"Training complete! Best Val Acc: {best_val_acc:.2f}%")
    return model

print("✓ Training functions loaded!")

In [ ]:
def evaluate_model_with_inference_time(model, data_loader, device, config, split_name="test"):
    """Evaluate model and measure inference time"""
    model.eval()

    all_predictions = []
    all_targets = []
    all_probabilities = []
    total_loss = 0.0
    num_batches = 0

    # Inference time measurement
    inference_times = []

    criterion = model.get_loss_function()

    with torch.no_grad():
        for data, target in tqdm(data_loader, desc=f"Evaluating {split_name}", leave=False):
            data, target = data.to(device), target.to(device).long()

            # Measure inference time
            if device.type == 'cuda':
                torch.cuda.synchronize()
            start_time = time.time()

            output = model(data, pretrain_mode=False)

            if device.type == 'cuda':
                torch.cuda.synchronize()
            end_time = time.time()

            batch_time = (end_time - start_time) * 1000  # Convert to ms
            inference_times.append(batch_time / data.size(0))  # Per image

            loss = criterion(output, target)
            total_loss += loss.item()
            num_batches += 1

            probs = torch.softmax(output, dim=1)
            _, predicted = torch.max(output, 1)

            all_probabilities.extend(probs.cpu().numpy())
            all_predictions.extend(predicted.cpu().numpy())
            all_targets.extend(target.cpu().numpy())

    all_predictions = np.array(all_predictions)
    all_targets = np.array(all_targets)
    all_probabilities = np.array(all_probabilities)

    avg_loss = total_loss / num_batches if num_batches > 0 else float('inf')
    accuracy = np.mean(all_predictions == all_targets)

    # Inference time stats
    avg_inference_time = np.mean(inference_times)
    std_inference_time = np.std(inference_times)

    # Confusion matrix
    cm = confusion_matrix(all_targets, all_predictions)

    # Per-class metrics
    per_class_metrics = {}
    for i, class_name in enumerate(config.class_names):
        tp = cm[i, i]
        fp = cm[:, i].sum() - tp
        fn = cm[i, :].sum() - tp
        tn = cm.sum() - tp - fp - fn

        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        f1 = 2 * (precision * sensitivity) / (precision + sensitivity) if (precision + sensitivity) > 0 else 0

        per_class_metrics[class_name] = {
            'sensitivity': sensitivity,
            'specificity': specificity,
            'precision': precision,
            'f1_score': f1
        }

    # ROC-AUC
    try:
        y_bin = label_binarize(all_targets, classes=range(config.num_classes))
        roc_auc = roc_auc_score(y_bin, all_probabilities, average='macro', multi_class='ovr')
    except:
        roc_auc = 0

    macro_f1 = np.mean([m['f1_score'] for m in per_class_metrics.values()])
    macro_precision = np.mean([m['precision'] for m in per_class_metrics.values()])
    macro_recall = np.mean([m['sensitivity'] for m in per_class_metrics.values()])

    return {
        'accuracy': accuracy,
        'loss': avg_loss,
        'macro_f1': macro_f1,
        'macro_precision': macro_precision,
        'macro_recall': macro_recall,
        'roc_auc': roc_auc,
        'per_class_metrics': per_class_metrics,
        'confusion_matrix': cm,
        'predictions': all_predictions,
        'targets': all_targets,
        'probabilities': all_probabilities,
        'avg_inference_time_ms': avg_inference_time,
        'std_inference_time_ms': std_inference_time
    }

def print_detailed_results(results, split_name, backbone_name):
    print(f"\n{split_name.upper()} RESULTS - {backbone_name}")
    print("="*80)
    print(f"Accuracy:        {results['accuracy']:.4f}")
    print(f"Macro F1:        {results['macro_f1']:.4f}")
    print(f"Macro Precision: {results['macro_precision']:.4f}")
    print(f"Macro Recall:    {results['macro_recall']:.4f}")
    print(f"ROC-AUC:         {results['roc_auc']:.4f}")
    print(f"Inference Time:  {results['avg_inference_time_ms']:.2f} ± {results['std_inference_time_ms']:.2f} ms/image")

print("✓ Evaluation functions with inference time loaded!")

In [ ]:
def plot_confusion_matrix(cm, class_names, backbone_name, quantum_name, save_path=None):
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.title(f'Confusion Matrix - {backbone_name} ({quantum_name})', fontsize=14, fontweight='bold')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()

def create_comparison_table(all_results, config):
    """Create comprehensive comparison table"""
    data = []
    for backbone_key, results in all_results.items():
        backbone_name = config.backbones[backbone_key]['name']
        test_results = results.get('test_results', {})

        row = {
            'Backbone': backbone_name,
            'Test Acc': f"{test_results.get('accuracy', 0):.4f}",
            'Test F1': f"{test_results.get('macro_f1', 0):.4f}",
            'Precision': f"{test_results.get('macro_precision', 0):.4f}",
            'Recall': f"{test_results.get('macro_recall', 0):.4f}",
            'ROC-AUC': f"{test_results.get('roc_auc', 0):.4f}",
            'Inference (ms)': f"{test_results.get('avg_inference_time_ms', 0):.2f}"
        }
        data.append(row)

    df = pd.DataFrame(data)
    df = df.sort_values('Test Acc', ascending=False)

    print(f"\n{'='*120}")
    print(f"RESULTS - {config.quantum_name}")
    print(f"{'='*120}")
    print(df.to_string(index=False))
    print(f"{'='*120}")

    csv_path = os.path.join(config.results_dir, f'{config.quantum_name}_comparison.csv')
    df.to_csv(csv_path, index=False)
    print(f"✓ Results saved to: {csv_path}")

    return df

def create_combined_comparison(results_Q4L2, results_Q6L3, config_Q4L2, config_Q6L3):
    """Create combined comparison across both quantum configs"""
    data = []

    all_backbones = set(list(results_Q4L2.keys()) + list(results_Q6L3.keys()))

    for backbone_key in all_backbones:
        backbone_name = config_Q4L2.backbones[backbone_key]['name']

        # Q4L2 results
        q4l2_results = results_Q4L2.get(backbone_key, {}).get('test_results', {})
        q4l2_acc = q4l2_results.get('accuracy', 0)
        q4l2_f1 = q4l2_results.get('macro_f1', 0)
        q4l2_time = q4l2_results.get('avg_inference_time_ms', 0)

        # Q6L3 results
        q6l3_results = results_Q6L3.get(backbone_key, {}).get('test_results', {})
        q6l3_acc = q6l3_results.get('accuracy', 0)
        q6l3_f1 = q6l3_results.get('macro_f1', 0)
        q6l3_time = q6l3_results.get('avg_inference_time_ms', 0)

        row = {
            'Backbone': backbone_name,
            'Q4L2 Acc': f"{q4l2_acc:.4f}",
            'Q4L2 F1': f"{q4l2_f1:.4f}",
            'Q4L2 Time(ms)': f"{q4l2_time:.2f}",
            'Q6L3 Acc': f"{q6l3_acc:.4f}",
            'Q6L3 F1': f"{q6l3_f1:.4f}",
            'Q6L3 Time(ms)': f"{q6l3_time:.2f}",
            'Acc Diff': f"{q6l3_acc - q4l2_acc:+.4f}",
            'Best Config': 'Q6L3' if q6l3_acc > q4l2_acc else 'Q4L2'
        }
        data.append(row)

    df = pd.DataFrame(data)
    df = df.sort_values('Q6L3 Acc', ascending=False)

    print(f"\n{'='*150}")
    print("COMBINED COMPARISON: 4Q_2L vs 6Q_3L")
    print(f"{'='*150}")
    print(df.to_string(index=False))
    print(f"{'='*150}")

    csv_path = os.path.join(config_Q4L2.results_dir.replace('4Q_2L', 'combined'), 'combined_comparison.csv')
    os.makedirs(os.path.dirname(csv_path), exist_ok=True)
    df.to_csv(csv_path, index=False)
    print(f"✓ Combined results saved to: {csv_path}")

    return df

print("✓ Visualization functions loaded!")

In [ ]:
def train_single_backbone(backbone_key, config, data_loaders, qnode):
    """Train a single backbone"""
    backbone_name = config.backbones[backbone_key]['name']

    print(f"\n{'='*80}")
    print(f"TRAINING: {backbone_name} with {config.quantum_name}")
    print(f"{'='*80}")

    # Initialize model
    model = HybridQNN(backbone_key, config, qnode)

    # Phase 1: Fine-tuning
    model = fine_tune_backbone(
        model,
        data_loaders['train_loader'],
        data_loaders['val_loader'],
        config,
        backbone_name
    )

    # Save pretrained
    pretrain_path = os.path.join(config.results_dir, f'{backbone_key}_pretrained.pth')
    torch.save(model.state_dict(), pretrain_path)

    # Phase 2: Hybrid QNN training
    model = train_hybrid_qnn(
        model,
        data_loaders['train_loader'],
        data_loaders['val_loader'],
        config,
        backbone_name
    )

    # Save final model
    final_path = os.path.join(config.results_dir, f'{backbone_key}_final.pth')
    torch.save(model.state_dict(), final_path)

    # Evaluate on test with inference time
    test_results = evaluate_model_with_inference_time(
        model, data_loaders['test_loader'],
        config.device, config, "test"
    )
    print_detailed_results(test_results, "TEST", backbone_name)

    # Plot confusion matrix
    cm_path = os.path.join(config.results_dir, f'{backbone_key}_confusion_matrix.png')
    plot_confusion_matrix(test_results['confusion_matrix'], config.class_names,
                         backbone_name, config.quantum_name, save_path=cm_path)

    # Clear memory
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {
        'backbone_key': backbone_key,
        'backbone_name': backbone_name,
        'test_results': test_results,
        'model': model
    }

def train_all_backbones_for_quantum_config(config, data_loaders):
    """Train all 11 backbones for one quantum configuration"""

    print(f"\n{'='*80}")
    print(f"TRAINING ALL 11 BACKBONES - {config.quantum_name}")
    print(f"{'='*80}")

    # Create quantum circuit for this config
    qnode = create_quantum_circuit(config)

    all_results = {}

    for backbone_key in config.backbones.keys():
        try:
            results = train_single_backbone(backbone_key, config, data_loaders, qnode)
            all_results[backbone_key] = results
        except Exception as e:
            print(f"\n Error training {backbone_key}: {e}")
            import traceback
            traceback.print_exc()
            continue

    return all_results

print("✓ Training pipeline loaded!")

In [ ]:
# Organize dataset ONCE (shared across both quantum configs)
print("\n" + "="*80)
print("ORGANIZING DATASET (SHARED)")
print("="*80)

# Use first quantum config for dataset organization
temp_config = Config(quantum_configs['Q4L2'])
success = organize_balanced_dataset(
    temp_config.original_dataset_path,
    temp_config.processed_dataset_path,
    temp_config
)

if success:
    print("\n Dataset successfully organized and ready for both experiments!")
else:
    print("\n Dataset organization failed!")

In [ ]:
print("\n" + "="*80)
print("EXPERIMENT 1: 4 QUBITS, 2 LAYERS")
print("="*80)

# Create config
config_Q4L2 = Config(quantum_configs['Q4L2'])

# Create dataloaders
data_loaders_Q4L2 = create_dataloaders(config_Q4L2)

# Train all backbones
start_time_Q4L2 = time.time()
results_Q4L2 = train_all_backbones_for_quantum_config(config_Q4L2, data_loaders_Q4L2)
total_time_Q4L2 = time.time() - start_time_Q4L2

# Create comparison table
comparison_df_Q4L2 = create_comparison_table(results_Q4L2, config_Q4L2)

# Find best model
if results_Q4L2:
    best_Q4L2 = max(results_Q4L2.items(),
                    key=lambda x: x[1]['test_results']['accuracy'])

    print(f"\n{'='*80}")
    print("BEST MODEL - 4Q_2L")
    print(f"{'='*80}")
    print(f"Backbone: {best_Q4L2[1]['backbone_name']}")
    print(f"Test Accuracy: {best_Q4L2[1]['test_results']['accuracy']:.4f}")
    print(f"Test F1: {best_Q4L2[1]['test_results']['macro_f1']:.4f}")
    print(f"Inference Time: {best_Q4L2[1]['test_results']['avg_inference_time_ms']:.2f} ms")
    print(f"\nTotal Time: {total_time_Q4L2/60:.1f} minutes")
    print(f"{'='*80}")

# Clear memory
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("\nExperiment 1 complete!")

In [ ]:
print("\n" + "="*80)
print("EXPERIMENT 2: 6 QUBITS, 3 LAYERS")
print("="*80)

# Create config
config_Q6L3 = Config(quantum_configs['Q6L3'])

# Create dataloaders (reuse same processed dataset)
data_loaders_Q6L3 = create_dataloaders(config_Q6L3)

# Train all backbones
start_time_Q6L3 = time.time()
results_Q6L3 = train_all_backbones_for_quantum_config(config_Q6L3, data_loaders_Q6L3)
total_time_Q6L3 = time.time() - start_time_Q6L3

# Create comparison table
comparison_df_Q6L3 = create_comparison_table(results_Q6L3, config_Q6L3)

# Find best model
if results_Q6L3:
    best_Q6L3 = max(results_Q6L3.items(),
                    key=lambda x: x[1]['test_results']['accuracy'])

    print(f"\n{'='*80}")
    print("BEST MODEL - 6Q_3L")
    print(f"{'='*80}")
    print(f"Backbone: {best_Q6L3[1]['backbone_name']}")
    print(f"Test Accuracy: {best_Q6L3[1]['test_results']['accuracy']:.4f}")
    print(f"Test F1: {best_Q6L3[1]['test_results']['macro_f1']:.4f}")
    print(f"Inference Time: {best_Q6L3[1]['test_results']['avg_inference_time_ms']:.2f} ms")
    print(f"\nTotal Time: {total_time_Q6L3/60:.1f} minutes")
    print(f"{'='*80}")

print("\nExperiment 2 complete!")

In [ ]:
print("\n" + "="*80)
print("FINAL COMBINED ANALYSIS")
print("="*80)

# Create combined comparison
combined_df = create_combined_comparison(results_Q4L2, results_Q6L3, config_Q4L2, config_Q6L3)

# Summary statistics
print(f"\n{'='*80}")
print("SUMMARY STATISTICS")
print(f"{'='*80}")

# Best overall model
all_models = []
for backbone_key, results in results_Q4L2.items():
    all_models.append({
        'backbone': config_Q4L2.backbones[backbone_key]['name'],
        'config': '4Q_2L',
        'accuracy': results['test_results']['accuracy'],
        'f1': results['test_results']['macro_f1'],
        'inference_time': results['test_results']['avg_inference_time_ms']
    })

for backbone_key, results in results_Q6L3.items():
    all_models.append({
        'backbone': config_Q6L3.backbones[backbone_key]['name'],
        'config': '6Q_3L',
        'accuracy': results['test_results']['accuracy'],
        'f1': results['test_results']['macro_f1'],
        'inference_time': results['test_results']['avg_inference_time_ms']
    })

best_overall = max(all_models, key=lambda x: x['accuracy'])

print(f"\nBEST OVERALL MODEL:")
print(f"  Backbone: {best_overall['backbone']}")
print(f"  Quantum Config: {best_overall['config']}")
print(f"  Test Accuracy: {best_overall['accuracy']:.4f}")
print(f"  Test F1: {best_overall['f1']:.4f}")
print(f"  Inference Time: {best_overall['inference_time']:.2f} ms/image")

# Average performance
q4l2_avg_acc = np.mean([r['test_results']['accuracy'] for r in results_Q4L2.values()])
q6l3_avg_acc = np.mean([r['test_results']['accuracy'] for r in results_Q6L3.values()])

print(f"\nAVERAGE PERFORMANCE:")
print(f"  4Q_2L: {q4l2_avg_acc:.4f}")
print(f"  6Q_3L: {q6l3_avg_acc:.4f}")
print(f"  Improvement: {(q6l3_avg_acc - q4l2_avg_acc):.4f} ({((q6l3_avg_acc - q4l2_avg_acc)/q4l2_avg_acc * 100):+.2f}%)")

# Save final summary
final_summary = {
    'experiment': 'Transfer Learning + Hybrid QNN - 11 Backbones × 2 Quantum Configs',
    'best_overall': {
        'backbone': best_overall['backbone'],
        'quantum_config': best_overall['config'],
        'test_accuracy': float(best_overall['accuracy']),
        'test_f1': float(best_overall['f1']),
        'inference_time_ms': float(best_overall['inference_time'])
    },
    'average_performance': {
        '4Q_2L': float(q4l2_avg_acc),
        '6Q_3L': float(q6l3_avg_acc),
        'improvement': float(q6l3_avg_acc - q4l2_avg_acc)
    },
    'total_time_hours': {
        '4Q_2L': total_time_Q4L2 / 3600,
        '6Q_3L': total_time_Q6L3 / 3600,
        'total': (total_time_Q4L2 + total_time_Q6L3) / 3600
    }
}

summary_path = 'final_experiment_summary.json'
with open(summary_path, 'w') as f:
    json.dump(final_summary, f, indent=2)

print(f"\n✓ Final summary saved to: {summary_path}")

print(f"\n{'='*80}")
print("ALL EXPERIMENTS COMPLETE!")
print(f"{'='*80}")
print(f"Total experiment time: {(total_time_Q4L2 + total_time_Q6L3)/3600:.2f} hours")
print(f"Results saved in:")
print(f"  - {config_Q4L2.results_dir}/")
print(f"  - {config_Q6L3.results_dir}/")
print(f"{'='*80}")